# Dataset Manifest Generator and Prepare Dataset

This notebook generates a manifest for the full train dataset. Each row maps one image to its label, batch, and split. Labels support one or more percentages, for example `R5` or `R5 S10` (rice hull 5%, salt 10%) and then prepares and preprocess the dataset for it to stay in the disk for future retraining.

Keep the same output keys throughout one training run. For rice hull and salt, use `R5 S0` when salt is absent. Output keys are sorted consistently, so `R5 S10` and `S10 R5` both map to `[5, 10]` in `R, S` order. Keep the original folder label unchanged for prepared-image paths.

## Manifest format

| image_path                      | label | batch   | split      |
|:--------------------------------|:-----:|:--------|:----------:|
| dataset/R5/batch_1/img_001.jpg  | R5    | batch_1 | training   |
| dataset/R0/batch_2/img_001.jpg  | R0    | batch_2 | testing    |
| dataset/R10/batch_3/img_001.jpg | R10   | batch_3 | validation |

In [1]:

import pandas as pd

import config
from utils.labels import infer_label_keys
from utils.prepare_dataset import prepare_dataset

config.check_config(is_notebook=True)

CHECK CONFIGURATION DETAILS:
RAW DATASET DIR: C:\Users\realb\Documents\PROJECT_FILES\Dataset\raw_material_batched
TASK: classification
MANIFEST FILENAME (same dir as this file): C:\Users\realb\Documents\PROJECT_FILES\DABARResearchTraining\raw_material_manifest_classification.csv
PREPROCESSED DATASET DIR: C:\Users\realb\Documents\PROJECT_FILES\Dataset\for_training\classification\raw_material_batched
PTH MODEL OUTPUT: C:\Users\realb\Documents\PROJECT_FILES\DABARResearchTraining\models\pth\auto.pth
ONNX MODEL OUTPUT: C:\Users\realb\Documents\PROJECT_FILES\DABARResearchTraining\models\onnx\auto.onnx
EPOCHS: 300
BATCH SIZE: 32
Review the configuration above before running the next cell.


Records the ```image_path```, ```label```, and ```batch```

In [2]:
records = []

for image in config.RAW_DATASET_DIR.rglob("*"):
    if not image.is_file():
        continue
    if image.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
        continue

    records.append({
        "image_path": str(image),
        "batch": image.parent.name,
        "label": image.parent.parent.name,
    })
print(records[:5])
if not records:
    raise ValueError("IMAGE WHEREE???????")

df = pd.DataFrame(records)

[{'image_path': 'C:\\Users\\realb\\Documents\\PROJECT_FILES\\Dataset\\raw_material_batched\\soybean_meal\\batch_1\\im_20260331_1608060x1ag.jpg', 'batch': 'batch_1', 'label': 'soybean_meal'}, {'image_path': 'C:\\Users\\realb\\Documents\\PROJECT_FILES\\Dataset\\raw_material_batched\\soybean_meal\\batch_1\\im_20260331_1608060x1bg.jpg', 'batch': 'batch_1', 'label': 'soybean_meal'}, {'image_path': 'C:\\Users\\realb\\Documents\\PROJECT_FILES\\Dataset\\raw_material_batched\\soybean_meal\\batch_1\\im_20260331_1608060x1cg.jpg', 'batch': 'batch_1', 'label': 'soybean_meal'}, {'image_path': 'C:\\Users\\realb\\Documents\\PROJECT_FILES\\Dataset\\raw_material_batched\\soybean_meal\\batch_1\\im_20260331_161021(nil)g.jpg', 'batch': 'batch_1', 'label': 'soybean_meal'}, {'image_path': 'C:\\Users\\realb\\Documents\\PROJECT_FILES\\Dataset\\raw_material_batched\\soybean_meal\\batch_1\\im_20260331_161207(nil)g.jpg', 'batch': 'batch_1', 'label': 'soybean_meal'}]


Splits the batches in their ```split``` calculated 70% training 15% testing 15% validation, splitted by batch count, not image count

In [3]:
if config.TASK == "classification":
    # Keep the complete folder names as classes.
    class_names = sorted(df["label"].unique())
    print("Classes:", class_names)
else:
    # Parse labels such as R5 S0 into regression outputs.
    label_keys = infer_label_keys(df["label"])
    print("Output order:", label_keys)
    
print(df.head())

Classes: ['copra_meal', 'corn', 'rice_bran', 'soybean_meal']
                                          image_path    batch         label
0  C:\Users\realb\Documents\PROJECT_FILES\Dataset...  batch_1  soybean_meal
1  C:\Users\realb\Documents\PROJECT_FILES\Dataset...  batch_1  soybean_meal
2  C:\Users\realb\Documents\PROJECT_FILES\Dataset...  batch_1  soybean_meal
3  C:\Users\realb\Documents\PROJECT_FILES\Dataset...  batch_1  soybean_meal
4  C:\Users\realb\Documents\PROJECT_FILES\Dataset...  batch_1  soybean_meal


In [4]:
batches = df[["label", "batch"]].drop_duplicates().copy()
batches["split"] = "" # split are training, validation, and testing

for label in batches["label"].unique():
    mask = batches["label"] == label 
    group = batches[mask].sample(frac=1, random_state=42)
    
    batches_per_label = len(group)
    
    if batches_per_label < 3:
        raise ValueError(f"Not enough batches for label '{label}' to split into training, validation, and testing.")
    
    train_batches = min(round(batches_per_label * 0.7), batches_per_label - 2)
    remaining_batches = batches_per_label - train_batches
    validation_batches = remaining_batches // 2
    test_batches = remaining_batches - validation_batches
    
    print(f"Label: {label}, Total Batches: {batches_per_label}, Train: {train_batches}, Validation: {validation_batches}, Test: {test_batches}")
    
    batches.loc[group.index, "split"] = (
        ["training"] * train_batches
        + ["validation"] * validation_batches
        + ["testing"] * test_batches
    )
    
df = df.merge(batches, on=["label", "batch"], validate="many_to_one")

Label: soybean_meal, Total Batches: 7, Train: 5, Validation: 1, Test: 1
Label: rice_bran, Total Batches: 7, Train: 5, Validation: 1, Test: 1
Label: corn, Total Batches: 7, Train: 5, Validation: 1, Test: 1
Label: copra_meal, Total Batches: 7, Train: 5, Validation: 1, Test: 1


In [5]:
print(pd.crosstab(batches["label"], batches["split"]))
print(pd.crosstab(df["label"], df["split"]))

print(config.MANIFEST_FILENAME)
df.to_csv(config.MANIFEST_FILENAME, index=False)

split         testing  training  validation
label                                      
copra_meal          1         5           1
corn                1         5           1
rice_bran           1         5           1
soybean_meal        1         5           1
split         testing  training  validation
label                                      
copra_meal         10        50          10
corn               10        50          10
rice_bran          10        50          10
soybean_meal       10        50          10
C:\Users\realb\Documents\PROJECT_FILES\DABARResearchTraining\raw_material_manifest_classification.csv


### Prepare and preprocess dataset

In [ ]:
prepare_dataset(config.MANIFEST_FILENAME, config.PREPROCESSED_DATASET_DIR)

1/280: SKIPPED im_20260331_1608060x1ag.jpg
2/280: SKIPPED im_20260331_1608060x1bg.jpg
3/280: SKIPPED im_20260331_1608060x1cg.jpg
4/280: SKIPPED im_20260331_161021(nil)g.jpg
5/280: SKIPPED im_20260331_161207(nil)g.jpg
6/280: SKIPPED im_20260331_161359(nil)g.jpg
7/280: SKIPPED im_20260331_161605(nil)g.jpg
8/280: SKIPPED im_20260331_161824(nil)g.jpg
9/280: SKIPPED im_20260331_162020(nil)g.jpg
10/280: SKIPPED im_20260331_162237(nil)g.jpg
11/280: SKIPPED im_20260331_1608060x1dg.jpg
12/280: SKIPPED im_20260331_1608060x1eg.jpg
13/280: SKIPPED im_20260331_1608060x1fg.jpg
14/280: SKIPPED im_20260331_1608060x1g.jpg
15/280: SKIPPED im_20260331_1608060x2ag.jpg
16/280: SKIPPED im_20260331_1608060x2bg.jpg
17/280: SKIPPED im_20260331_1608060x2cg.jpg
18/280: SKIPPED im_20260331_1608060x2dg.jpg
19/280: SKIPPED im_20260331_1608060x2eg.jpg
20/280: SKIPPED im_20260331_1608060x2fg.jpg
21/280: SKIPPED im_20260331_1608060x10g.jpg
22/280: SKIPPED im_20260331_1608060x11g.jpg
23/280: SKIPPED im_20260331_1608060